# Jamii Afya Falcon production pipeline

This notebook is audit-first and resume-first. It does not launch a long training run unless `FALCON_RUN_MODE=stage` is explicitly set. Every child command is streamed line-by-line, and the trainer writes JSONL events, metrics, heartbeats, and Trainer checkpoints.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
EXP = REPO / 'experiments' / 'falcon-production-v1'
LOG = EXP / 'kaggle-logs'
BRANCH = 'research/edge35-adaptive-streaming'
RUN_MODE = os.environ.get('FALCON_RUN_MODE', 'audit')
STAGE = os.environ.get('FALCON_STAGE', 'stage_a_capability_preserving')
CHECKPOINT_DATASET = os.environ.get('FALCON_CHECKPOINT_DATASET', '')
INIT_ADAPTER = os.environ.get('FALCON_INIT_ADAPTER', '')
LOG.mkdir(parents=True, exist_ok=True)
def streamed(command, cwd=REPO, name='command.log', env=None):
    path = LOG / name
    merged = os.environ.copy(); merged.update(env or {}); merged['PYTHONUNBUFFERED'] = '1'
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with path.open('a', encoding='utf-8', buffering=1) as handle:
        proc = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            rendered = f'[{stamp}] {line}'
            print(rendered, end='', flush=True); handle.write(rendered)
        code = proc.wait()
        print(f'EXIT={code}', flush=True); handle.write(f'EXIT={code}\n')
    if code: raise RuntimeError(f'command failed: {command}')
if not (REPO / '.git').exists():
    if REPO.exists(): shutil.rmtree(REPO)
    streamed(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
else:
    # Kaggle may reuse a worker checkout; never trust its previous branch/SHA.
    streamed(['git','-C',str(REPO),'fetch','origin',BRANCH], cwd=WORK, name='git-refresh.log')
    streamed(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH], cwd=WORK, name='git-refresh.log')
print('repo sha:', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'], text=True, capture_output=True, check=True).stdout.strip(), flush=True)
if not (REPO / 'requirements-falcon-production.txt').exists(): raise RuntimeError('clean checkout is missing pinned production requirements')
streamed([sys.executable,'-m','pip','install','-q','-r','requirements-falcon-production.txt'], name='pip.log')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
if 'P100' in gpu:
    # P100/sm_60 cannot use the image's current Torch or bitsandbytes path.
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','numpy<2'], name='numpy-p100.log')
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','torch==2.2.2','--index-url','https://download.pytorch.org/whl/cu118'], name='torch-p100.log')
    streamed([sys.executable,'-m','pip','uninstall','-y','torchao','torchvision','torchaudio','bitsandbytes'], name='optional-uninstall.log')
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-deps','transformers==4.53.3','tokenizers==0.21.4','peft==0.15.2','accelerate==1.7.0'], name='hf-stack-final.log')
else:
    streamed([sys.executable,'-m','pip','install','-q','bitsandbytes==0.50.2'], name='bitsandbytes.log')
print(json.dumps({'run_mode': RUN_MODE, 'stage': STAGE, 'gpu': gpu, 'checkpoint_dataset_configured': bool(CHECKPOINT_DATASET), 'repo': str(REPO)}, indent=2), flush=True)

In [ ]:
DATA_DIR = EXP / 'data'
MCQA = REPO / 'output' / 'accuracy_sft.jsonl'
if not MCQA.exists():
    # Public TRAIN splits only; the cap is explicit so the first clean-worker
    # audit is bounded. Increase only after measured throughput/token-share evidence.
    cap = os.environ.get('FALCON_MCQA_MAX_PER_DATASET', '2000')
    streamed([sys.executable, '-u', 'scripts/build_accuracy_sft.py', '--datasets', 'arc_easy', 'arc_challenge', 'openbookqa', 'mmlu_aux', 'medmcqa', 'medqa', 'pubmedqa', 'headqa', '--max-per-dataset', cap, '--out', str(MCQA)], name='mcqa-build.log')
streamed([sys.executable, '-u', 'scripts/build_falcon_dataset.py', '--config', 'configs/falcon-production-v1.json', '--out-dir', str(DATA_DIR)], name='dataset-build.log')
manifest = json.loads((DATA_DIR / 'data_manifest.json').read_text())
print(json.dumps({k: manifest[k] for k in ('counts','token_totals','token_shares_percent','facets','missing_sources')}, indent=2), flush=True)
if RUN_MODE == 'audit':
    print('AUDIT_ONLY: no training launched. Set FALCON_RUN_MODE=resume_test or stage after persistence is configured.', flush=True)

In [ ]:
if RUN_MODE == 'resume_test':
    # This is intentionally tiny and ephemeral: it verifies Trainer checkpoint
    # state/resume mechanics before any expensive production stage.
    test_run = EXP / 'resume-test'
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', 'configs/falcon-production-v1.json', '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '2', '--save-steps', '1', '--allow-ephemeral'], name='resume-test-first.log')
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', 'configs/falcon-production-v1.json', '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '4', '--save-steps', '1', '--resume-from-checkpoint', 'latest', '--allow-ephemeral'], name='resume-test-resume.log')
    ckpt_root = test_run / STAGE / 'checkpoints'
    first = json.loads((ckpt_root / 'checkpoint-2' / 'trainer_state.json').read_text())
    resumed = json.loads((ckpt_root / 'checkpoint-4' / 'trainer_state.json').read_text())
    assert first['global_step'] == 2 and resumed['global_step'] == 4
    assert any((ckpt_root / 'checkpoint-2').glob('optimizer.*')) and any((ckpt_root / 'checkpoint-2').glob('scheduler.*'))
    print('RESUME_TEST_PASS: global_step 2 -> 4 and optimizer/scheduler state files present.', flush=True)


In [ ]:
if RUN_MODE == 'stage':
    if not CHECKPOINT_DATASET:
        raise RuntimeError('Set FALCON_CHECKPOINT_DATASET to an existing private Kaggle dataset before a production stage.')
    streamed(['kaggle', 'datasets', 'files', '-d', CHECKPOINT_DATASET], cwd=REPO, name='checkpoint-dataset-preflight.log')
    run_dir = EXP / 'runs' / (os.environ.get('FALCON_RUN_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    env = {'FALCON_CHECKPOINT_DATASET': CHECKPOINT_DATASET}
    command = [sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', 'configs/falcon-production-v1.json', '--data-dir', str(DATA_DIR), '--run-dir', str(run_dir), '--stage', STAGE]
    if os.environ.get('FALCON_RESUME'):
        command += ['--resume-from-checkpoint', os.environ['FALCON_RESUME']]
    if INIT_ADAPTER:
        command += ['--init-adapter', INIT_ADAPTER]
    streamed(command, env=env, name=f'{STAGE}.log')
    adapter = run_dir / STAGE / 'checkpoints' / 'final-adapter'
    if adapter.exists():
        eval_dir = run_dir / STAGE / 'stage-eval'
        eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', 'configs/falcon-production-v1.json', '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(adapter), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '200')]
        for battery in os.environ.get('FALCON_STAGE_BATTERIES', '').split(','):
            if battery.strip(): eval_cmd += ['--battery', battery.strip()]
        streamed(eval_cmd, name=f'{STAGE}-eval.log')
else:
    print('No training stage requested.', flush=True)